# OmniVoice Yoruba — checkpoint verification (variance + cloning collapse)

**Cheap T4/L4 follow-up to the `full` finetune (no A100).** Loads `checkpoint-5000` from S3 and answers
two open questions the training run left:

1. **Eval variance / error bars** — masked-diffusion synthesis is stochastic (the unmask/position
   ordering samples from the global torch RNG), so a single eval pass is noisy (`tone_i2` bounced
   0.62↔0.68). We run the **same nb01 gate K×** with different seeds → **mean ± std (+ SEM)** for
   `tone_i2 / cer / ssim / i1_acc`.
2. **Cloning-collapse (§12B, fixed)** — 19 epochs on a ~single-speaker corpus can make the model
   forget how to clone *other* voices. We clone **held-out non-bible** voices and check
   `SSIM→that-voice` beats `SSIM→bible`. The original §12B silently found nothing (wrong manifest
   field names) — here we **self-diagnose the manifest schema** and **fall back to SLR86 speakers**.

Optional §9: if you uploaded `checkpoint-4500/3500/…` too, a one-pass **checkpoint-selection** table.

**GPU: T4 (16 GB) or L4 — inference only.** `hf-xet` removed (load-hang gotcha). The model load is made
**fully local** by borrowing the tokenizer + codec from the base OmniVoice snapshot (a training
checkpoint may omit them).

## 1. Install + restart (numpy<2, omnivoice, gate stack; remove hf-xet)

In [ ]:
%pip install --quiet "numpy<2"
%pip install --quiet omnivoice
%pip install --quiet boto3 soundfile soxr librosa speechbrain "swift-f0" pyworld accelerate safetensors "huggingface_hub>=0.24.0" tqdm
%pip uninstall -y hf-xet
import os
print("Installs done. RESTARTING so numpy<2 takes effect — run from the NEXT cell.", flush=True)
os._exit(0)

In [ ]:
import numpy as np
assert np.__version__.startswith("1."), "numpy<2 pin did not take — re-run install + restart"
print("numpy", np.__version__, "OK")

## 2. Clone OmniVoice (introspect API) + `mosesdaudu001/tone-on-a-budget` (approachB gate)

In [ ]:
import os, subprocess, sys, shutil
OV_DIR = "/content/OmniVoice" if os.path.isdir("/content") else "/tmp/OmniVoice"
if os.path.exists(OV_DIR): shutil.rmtree(OV_DIR)
subprocess.run(["git","clone","--depth","1","https://github.com/k2-fsa/OmniVoice.git",OV_DIR],check=True)
tsv = os.path.join(OV_DIR,"docs","lang_id_name_map.tsv"); yo_ok=False
if os.path.exists(tsv):
    for ln in open(tsv):
        c=ln.rstrip("\n").split("\t")
        if (c and c[0]=="yo") or (len(c)>1 and c[1].lower()=="yoruba"):
            print("Yoruba lang line:",ln.rstrip()); yo_ok=(c[0]=="yo")
assert yo_ok, "Yoruba code is not 'yo' — fix LANG_CODE before synthesizing"
print(subprocess.run(["grep","-n","-A","16","def generate",
      os.path.join(OV_DIR,"omnivoice","models","omnivoice.py")],capture_output=True,text=True).stdout[:900])

In [ ]:
# tone-metric package (public) — replaces the old git clone + sys.path of tone_metric/
%pip install --quiet --no-deps "git+https://github.com/mosesdaudu001/tone-on-a-budget.git"
import os, subprocess, sys, shutil, importlib
importlib.invalidate_caches()
import tone_metric
from tone_metric import tone_probe as tp
from tone_metric import tone_f0_abs as f0a
from tone_metric.grpo_reward import RewardModels
CODE_DIR = os.path.dirname(tone_metric.__file__)   # minimal_pairs_draft.json ships as package data
print("tone_metric", tone_metric.__version__, "from", CODE_DIR)


## 3. Secrets + S3 + constants + Xet env + pre-fetch the base OmniVoice snapshot

The base snapshot bundles BOTH the text tokenizer and the audio codec (`audio_tokenizer/`); §4 copies
whatever the training checkpoint is missing from it, so the load is fully local.

In [ ]:
import os, getpass, boto3, random, torch, numpy as np, shutil, glob
def _secret(k):
    try:
        from google.colab import userdata
        v=userdata.get(k)
        if v: return v
    except Exception: pass
    return os.environ.get(k) or getpass.getpass(f"{k}: ")
os.environ["AWS_ACCESS_KEY_ID"]=_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"]=_secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]=os.environ.get("AWS_DEFAULT_REGION","us-east-1")
HF_TOKEN=_secret("HF_TOKEN"); os.environ["HF_TOKEN"]=HF_TOKEN or ""
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN)
os.environ["HF_HUB_DISABLE_XET"]="1"; os.environ["HF_HUB_ENABLE_HF_TRANSFER"]="0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"]="60"; os.environ["HF_HUB_ETAG_TIMEOUT"]="60"
try:
    import huggingface_hub.constants as _hc; _hc.HF_HUB_DISABLE_XET=True
except Exception: pass

BUCKET="codec-audio-data"
s3=boto3.client("s3",region_name=os.environ["AWS_DEFAULT_REGION"]); s3.head_bucket(Bucket=BUCKET)
print("S3 connected:",BUCKET)

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
SR=24000; LANG_CODE="yo"; K_RUNS=5; VARIANCE_SEEDS=[4242,1234,2026,777,99]; N_LONG=8
CKPT_PREFIX="tts_checkpoints/omnivoice_yoruba/full"
BIBLE_MANIFEST="tts_data/yoruba_gold/s1_train.v2.jsonl"
HOLDOUTS_KEY="tts_data/yoruba_gold/holdouts.v1.json"
S3_TONE_PREFIX="tts_data/yoruba/tone_v2"; F0CAL_KEY=f"{S3_TONE_PREFIX}/f0_abs_calibration.v1.json"
NV_MANIFEST="tts_data/yoruba/manifest.v3.jsonl"
OUT_PREFIX="tts_data/yoruba/omnivoice_ft_verify/full"
WORK="/content/ov_verify" if os.path.isdir("/content") else "/tmp/ov_verify"; os.makedirs(WORK,exist_ok=True)

from huggingface_hub import snapshot_download
HUB=os.path.join(os.path.expanduser(os.environ.get("HF_HOME","~/.cache/huggingface")),"hub")
if os.path.isdir(os.path.join(HUB,".locks")): shutil.rmtree(os.path.join(HUB,".locks"),ignore_errors=True)
OV_BASE=None
for _k in range(3):
    try: OV_BASE=snapshot_download("k2-fsa/OmniVoice",max_workers=1,etag_timeout=60); break
    except Exception as e: print("base prefetch retry:",type(e).__name__,e)
assert OV_BASE and os.path.isdir(os.path.join(OV_BASE,"audio_tokenizer")), "could not pre-fetch k2-fsa/OmniVoice (tokenizer+codec source)"
print("base snapshot:",OV_BASE,"| DEVICE",DEVICE,"| K_RUNS",K_RUNS)

## 4. Download `checkpoint-5000` from S3 + load (fully local)

`§8` uploaded the final checkpoint's files **directly** under `CKPT_PREFIX/`. We pull only the
top-level files (non-empty tail, no further `/` → skips folder markers and any `checkpoint-*/` subdirs),
borrow tokenizer/codec from the base snapshot if missing, and load.

In [ ]:
import os, shutil
def s3_ls(prefix):
    out,tok=[],None
    while True:
        kw=dict(Bucket=BUCKET,Prefix=prefix)
        if tok: kw["ContinuationToken"]=tok
        r=s3.list_objects_v2(**kw); out+=[(o["Key"],o["Size"]) for o in r.get("Contents",[])]
        if r.get("IsTruncated"): tok=r.get("NextContinuationToken")
        else: break
    return out

AUX=["tokenizer.json","tokenizer_config.json","special_tokens_map.json","vocab.json",
     "merges.txt","chat_template.jinja","generation_config.json","added_tokens.json"]
def materialize_ckpt(local_dir):
    have=set(os.listdir(local_dir))
    for fn in AUX:
        if fn not in have and os.path.exists(os.path.join(OV_BASE,fn)):
            shutil.copy(os.path.join(OV_BASE,fn),os.path.join(local_dir,fn))
    at=os.path.join(local_dir,"audio_tokenizer")
    if not os.path.isdir(at): shutil.copytree(os.path.join(OV_BASE,"audio_tokenizer"),at)
    return local_dir

def pull_top_level(prefix,dest):
    os.makedirs(dest,exist_ok=True); n=0
    for k,sz in s3_ls(prefix+"/"):
        tail=k[len(prefix)+1:]
        if not tail or "/" in tail: continue        # skip folder markers + subdir files
        s3.download_file(BUCKET,k,os.path.join(dest,os.path.basename(k))); n+=1
    return n

allk=s3_ls(CKPT_PREFIX+"/")
assert allk, f"nothing under s3://{BUCKET}/{CKPT_PREFIX}/ — was §8 run?"
CKPT_LOCAL=os.path.join(WORK,"ckpt5000")
n=pull_top_level(CKPT_PREFIX,CKPT_LOCAL); print("downloaded",n,"top-level files ->",CKPT_LOCAL)
materialize_ckpt(CKPT_LOCAL)
files=set(os.listdir(CKPT_LOCAL))
assert "config.json" in files and any(f.endswith((".safetensors",".bin",".pt")) for f in files), f"no model: {sorted(files)}"
assert ("tokenizer.json" in files) or ("tokenizer_config.json" in files) or ("vocab.json" in files), f"no tokenizer even after base copy: {sorted(files)}"
SUBCKPTS=sorted({k[len(CKPT_PREFIX)+1:].split('/')[0] for k,_ in allk
                 if k[len(CKPT_PREFIX)+1:].startswith("checkpoint-")})
print("extra checkpoint subdirs on S3:", SUBCKPTS or "(none)")

import torch
from omnivoice import OmniVoice
ov=OmniVoice.from_pretrained(CKPT_LOCAL,device_map=("cuda:0" if DEVICE=="cuda" else "cpu"),dtype=torch.float16)
OV_SR=int(getattr(ov,"sampling_rate",0) or SR)
print("finetuned model loaded; sampling_rate =",OV_SR); assert OV_SR==24000

## 5. Gate stack (identical to nb01/nb03 §9)

In [ ]:
import json
from transformers import AutoModel, AutoFeatureExtractor
rm=RewardModels(device=DEVICE); assert rm.ecapa is not None, "ECAPA failed — SSIM needs it"
enc=AutoModel.from_pretrained("ajesujoba/AfriHuBERT").to(DEVICE).eval()
fe =AutoFeatureExtractor.from_pretrained("ajesujoba/AfriHuBERT")
objs=s3.list_objects_v2(Bucket=BUCKET,Prefix=f"{S3_TONE_PREFIX}/tone_probe_L").get("Contents")
assert objs, "no tone_probe_L* on S3 (run nb22)"
probe_key=sorted(o["Key"] for o in objs)[-1]
s3.download_file(BUCKET,probe_key,f"{WORK}/tone_probe.pt")
PROBE,PMETA=tp.load_probe(f"{WORK}/tone_probe.pt",DEVICE); LAYER=PMETA.get("layer",9)
s3.download_file(BUCKET,F0CAL_KEY,f"{WORK}/f0cal.json"); F0CAL=json.load(open(f"{WORK}/f0cal.json"))
I2_TH,I2_TL=F0CAL["theta_h"],F0CAL["theta_l"]; I2_MODE,I2_LATE=F0CAL.get("mode","blind"),F0CAL.get("late_frac",0.5)
print("probe:",probe_key,"| layer:",LAYER)

## 6. Probe set + clean bible reference (verbatim nb03 §9 (c))

In [ ]:
import io, json, random, soundfile as sf, numpy as np
MP=json.load(open(os.path.join(CODE_DIR,"minimal_pairs_draft.json")))
CARRIER=MP["carriers"][0]["template"]
probe_lines=[]
for s_ in MP["sets"]:
    for j,it in enumerate(s_["items"]):
        probe_lines.append(dict(id=f"mp_{s_['base']}_{j}",kind="minpair",base=s_["base"],
                                word=it["text"],tones=it.get("tones"),text=CARRIER.replace("___",it["text"])))
HOLD=json.loads(s3.get_object(Bucket=BUCKET,Key=HOLDOUTS_KEY)["Body"].read())
EVAL_ALL=list(HOLD["eval_texts"]); rng=random.Random(4242)
for k,e in enumerate(rng.sample(EVAL_ALL,N_LONG)):
    probe_lines.append(dict(id=f"long_{k}_{e['base']}",kind="long",base=e["base"],word=None,tones=None,text=e["text"]))
ref_row=None
for raw in io.TextIOWrapper(s3.get_object(Bucket=BUCKET,Key=BIBLE_MANIFEST)["Body"],encoding="utf-8"):
    r=json.loads(raw)
    if r.get("source")=="bible" and 3.0<=float(r.get("duration_sec",0))<=10.0: ref_row=r; break
assert ref_row is not None, "no bible ref row"
REF_WAV_PATH=f"{WORK}/ref.wav"; s3.download_file(BUCKET,ref_row["audio_s3_key"],REF_WAV_PATH)
REF_TEXT=ref_row["text"]; ref_wav,ref_sr=sf.read(REF_WAV_PATH,dtype="float32")
if ref_wav.ndim>1: ref_wav=ref_wav.mean(-1)
print("probe lines:",len(probe_lines),"| bible ref:",ref_row["audio_s3_key"])

## 7. Eval variance ×K (mean ± std, + SEM = std/√N)

Only `torch.manual_seed` drives `generate` (token class selection is greedy, position/unmask order is
sampled), so per-seed variance is real but may be modest; `tone_i2` (balanced over pooled TBUs) is the
headline metric.

In [ ]:
import soundfile as sf, numpy as np, random as _random
from collections import defaultdict
from tqdm.auto import tqdm

def _bal_tone_acc(pairs):
    if not pairs: return float("nan")
    tot,cor=defaultdict(int),defaultdict(int)
    for pp,tt in pairs: tot[tt]+=1; cor[tt]+=int(pp==tt)
    recs=[cor[c]/tot[c] for c in tot if tot[c]>0]
    return float(np.mean(recs)) if recs else float("nan")

def score_clip(wav,fs,text):
    logits,n16=rm.asr_logits(wav,fs)
    cer=RewardModels.cer(text,rm.decode_logits(logits))
    i1=tp.probe_score(wav,fs,text,PROBE,enc,fe,asr=rm.asr,proc=rm.asr_proc,device=DEVICE,
                      emissions=logits,n16=n16,layer=LAYER)
    i2=f0a.f0_abs_score(wav,fs,text,asr=rm.asr,proc=rm.asr_proc,device=DEVICE,emissions=logits,n16=n16,
                        theta_h=I2_TH,theta_l=I2_TL,mode=I2_MODE,mid_ref=None,late_frac=I2_LATE)
    pairs=[(pp,tt) for pp,tt in zip(i2["pred"],i2["target"]) if pp is not None]
    return dict(cer=cer,i1_acc=i1["accuracy"],i1_cov=i1["coverage"],
                tone_i2_cov=i2["coverage"],ssim=rm.ssim(wav,fs,ref_wav,ref_sr),
                len_ratio=(len(wav)/fs)/max(0.157*len(text),1e-6),pairs=pairs)

def one_pass(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    np.random.seed(seed); _random.seed(seed)
    rows=[]
    for p in probe_lines:
        a=ov.generate(text=p["text"],language=LANG_CODE,ref_audio=REF_WAV_PATH,ref_text=REF_TEXT)
        rows.append(dict(kind=p["kind"],**score_clip(np.asarray(a[0],dtype="float32"),OV_SR,p["text"])))
    allp=[pr for r in rows for pr in r["pairs"]]; v=lambda k:[r[k] for r in rows if r[k]==r[k]]
    return dict(seed=seed,tone_i2=_bal_tone_acc(allp),cer=float(np.mean(v("cer"))),
                ssim=float(np.mean(v("ssim"))),i1_acc=float(np.mean(v("i1_acc"))),
                tone_i2_cov=float(np.mean([r["tone_i2_cov"] for r in rows])),
                len_ratio=float(np.mean([r["len_ratio"] for r in rows])))

runs=[one_pass(s) for s in tqdm(VARIANCE_SEEDS,desc="variance passes")]
keys=["tone_i2","cer","ssim","i1_acc","tone_i2_cov","len_ratio"]
N=len(runs)
STAT={k:dict(mean=float(np.mean([r[k] for r in runs])),std=float(np.std([r[k] for r in runs])),
             sem=float(np.std([r[k] for r in runs])/max(N**0.5,1)),
             min=float(np.min([r[k] for r in runs])),max=float(np.max([r[k] for r in runs]))) for k in keys}
print(f"=== eval variance over {N} passes (mean ± std; sem=std/√N) ===")
for k in keys:
    print(f"  {k:12} {STAT[k]['mean']:.3f} ± {STAT[k]['std']:.3f} (sem {STAT[k]['sem']:.3f})  [{STAT[k]['min']:.3f},{STAT[k]['max']:.3f}]")
print("\nAnchors: zero-shot 0.598 | native ~0.58 | chance 0.333  (tone_i2)")
import json,datetime
s3.put_object(Bucket=BUCKET,Key=f"{OUT_PREFIX}/variance.json",
              Body=json.dumps(dict(date=datetime.date.today().isoformat(),seeds=VARIANCE_SEEDS,
                   per_pass=runs,stats=STAT,n_probe=len(probe_lines)),ensure_ascii=False,indent=2).encode())
print("-> s3://%s/%s/variance.json"%(BUCKET,OUT_PREFIX))

## 8. Cloning-collapse check — does the FT model still adopt a held-out non-bible voice?

Self-diagnosing: loads `manifest.v3.jsonl`, prints its real schema, resolves text/audio/speaker fields
by trying several candidate names, filters by held-out speakers (falls back to any usable rows), and
**falls back to SLR86 speakers** if NaijaVoices refs can't be resolved. Verdict gates on ≥2 voices AND
a per-voice majority (not just the pooled mean).

In [ ]:
import io, json, numpy as np, soundfile as sf
def _field(row,*cands):
    for c in cands:
        if row.get(c): return row[c]
    return None
def _load_ref(s3key):
    lp=f"{WORK}/clref_{os.path.basename(str(s3key))}"
    s3.download_file(BUCKET,s3key,lp)
    w,sr=sf.read(lp,dtype="float32"); w=w.mean(-1) if w.ndim>1 else w
    return lp,w,sr

VOICES=[]; src_used="none"
try:
    NV=[json.loads(l) for l in io.TextIOWrapper(
        s3.get_object(Bucket=BUCKET,Key=NV_MANIFEST)["Body"],encoding="utf-8") if l.strip()]
    print("NaijaVoices rows:",len(NV),"| first-row keys:",sorted(NV[0].keys()) if NV else "(empty)")
    heldset=set(HOLD.get("naijavoices_holdout_speakers",[])); print("holdout speakers listed:",len(heldset))
    def _usable(r): return _field(r,"ref_text","text","transcript") and _field(r,"ref_audio_key","audio_s3_key","wav_key")
    rows=[r for r in NV if _usable(r) and (not heldset or r.get("speaker_id") in heldset)]
    if not rows: rows=[r for r in NV if _usable(r)]
    seen=set()
    for r in rows:
        spk=_field(r,"speaker_id","spk","speaker") or "unk"
        if spk in seen: continue
        try:
            lp,w,sr=_load_ref(_field(r,"ref_audio_key","audio_s3_key","wav_key")); seen.add(spk)
            VOICES.append(dict(spk=spk,ref_path=lp,ref_text=_field(r,"ref_text","text","transcript"),ref_wav=w,ref_sr=sr))
        except Exception as e: print("  skip",spk,"->",type(e).__name__,e)
        if len(VOICES)>=3: break
    if VOICES: src_used="naijavoices"
except Exception as e:
    print("NaijaVoices path failed:",type(e).__name__,e)
if not VOICES:
    print("falling back to SLR86 speakers (distinct non-bible voices, but IN training -> weaker test)")
    try:
        seen=set()
        for raw in io.TextIOWrapper(s3.get_object(Bucket=BUCKET,Key=BIBLE_MANIFEST)["Body"],encoding="utf-8"):
            try: r=json.loads(raw)
            except Exception: continue
            if r.get("source")=="bible": continue
            spk=str(r.get("speaker_id","unk"))
            if spk in seen or not (3.0<=float(r.get("duration_sec",0))<=10.0): continue
            try:
                lp,w,sr=_load_ref(r["audio_s3_key"]); seen.add(spk)
                VOICES.append(dict(spk=spk,ref_path=lp,ref_text=r["text"],ref_wav=w,ref_sr=sr))
            except Exception as e: print("  skip",spk,"->",e)
            if len(VOICES)>=3: break
        if VOICES: src_used="slr86_fallback"
    except Exception as e: print("SLR86 fallback failed:",type(e).__name__,e)
print(f"\nresolved {len(VOICES)} held-out voice(s) | source = {src_used}")

NONBIBLE=None
if not VOICES:
    print("WARNING: no non-bible voice resolved -> cloning check skipped. Inspect schema above.")
else:
    test_lines=([p for p in probe_lines if p["kind"]=="long"][:3] or probe_lines[:3])
    rows=[]
    for v in VOICES:
        sv,sb=[],[]
        for p in test_lines:
            a=ov.generate(text=p["text"],language=LANG_CODE,ref_audio=v["ref_path"],ref_text=v["ref_text"])
            w=np.asarray(a[0],dtype="float32")
            sv.append(rm.ssim(w,OV_SR,v["ref_wav"],v["ref_sr"])); sb.append(rm.ssim(w,OV_SR,ref_wav,ref_sr))
        rows.append(dict(spk=v["spk"],to_voice=float(np.mean(sv)),to_bible=float(np.mean(sb))))
    print(f"\n{'held-out spk':20} {'SSIM->voice':>11} {'SSIM->bible':>11}")
    for r in rows:
        print(f"{str(r['spk'])[:20]:20} {r['to_voice']:>11.3f} {r['to_bible']:>11.3f}"
              f"{'  ok' if r['to_voice']>r['to_bible'] else '  <-- leaning to bible'}")
    mv=float(np.mean([r["to_voice"] for r in rows])); mb=float(np.mean([r["to_bible"] for r in rows]))
    n_ok=sum(1 for r in rows if r["to_voice"]>r["to_bible"])
    intact=bool(len(rows)>=2 and mv>mb+0.05 and n_ok>=(len(rows)+1)//2)
    NONBIBLE=dict(source=src_used,n_voices=len(rows),n_ok=n_ok,ssim_to_voice=mv,ssim_to_bible=mb,intact=intact,rows=rows)
    print(f"\nmean SSIM ->held-out {mv:.3f} vs ->bible {mb:.3f} | {n_ok}/{len(rows)} voices adopted -> "
          f"{'cloning diversity INTACT' if intact else 'WARNING: collapsing toward bible (or too few voices)'}")
    if src_used=="slr86_fallback":
        print("NOTE: SLR86 voices were IN training -> weaker (easier) test than true held-out NaijaVoices.")

import datetime
s3.put_object(Bucket=BUCKET,Key=f"{OUT_PREFIX}/capability_v2.json",
              Body=json.dumps(dict(date=datetime.date.today().isoformat(),variance_stats=STAT,
                   nonbible_clone=NONBIBLE),ensure_ascii=False,indent=2).encode())
print("-> s3://%s/%s/capability_v2.json"%(BUCKET,OUT_PREFIX))

## 8b. LISTEN — play + upload the cloning audio (the ear is the decisive collapse check)

SSIM is only a proxy; whether a clone actually sounds like the *target* speaker (vs the bible narrator)
is judged **by ear**. This plays each held-out target + its clones inline AND uploads them to
`…/omnivoice_ft_verify/full/listen/` for the native reviewer. Runs **before §9** (which frees the
model); if you run it later it reloads from the local checkpoint.

In [ ]:
# play + upload the cloning audio inline — the ear is the decisive collapse check (SSIM is only a proxy)
import numpy as np, soundfile as sf, os, torch
import IPython.display as ipd
from omnivoice import OmniVoice
if "ov" not in globals():     # §9 frees ov; reload from the on-disk local checkpoint (fast, no S3)
    ov = OmniVoice.from_pretrained(CKPT_LOCAL, device_map=("cuda:0" if DEVICE == "cuda" else "cpu"), dtype=torch.float16)
torch.manual_seed(4242)
_LL = [p for p in probe_lines if p["kind"] == "long"][:2]
_up = []
def _pu(arr, fs, tag):
    loc = f"{WORK}/listen_{tag}.wav"; sf.write(loc, np.asarray(arr, dtype="float32"), fs)
    s3.upload_file(loc, BUCKET, f"{OUT_PREFIX}/listen/{tag}.wav"); _up.append(tag)
    ipd.display(ipd.Audio(loc))
print("=== bible reference (the single voice the model was finetuned on) ===")
_pu(ref_wav, ref_sr, "ref_bible")
for v in (VOICES if "VOICES" in globals() else []):
    print(f"\n=== {v['spk']}: does the CLONE sound like THIS target, not the bible narrator? ===")
    print("target reference:"); _pu(v["ref_wav"], v["ref_sr"], f"{v['spk']}_target")
    for j, p in enumerate(_LL):
        a = ov.generate(text=p["text"], language=LANG_CODE, ref_audio=v["ref_path"], ref_text=v["ref_text"])
        print(f"clone {j}: {p['text'][:48]}"); _pu(a[0], OV_SR, f"{v['spk']}_clone{j}")
print(f"\nuploaded {len(_up)} wavs -> s3://{BUCKET}/{OUT_PREFIX}/listen/  (native ear is decisive)")

## 9. (Optional) Checkpoint selection — if you uploaded `checkpoint-4500/3500/…`

Frees the final model first (avoids a two-model + two-codec peak on a 16 GB T4), then evals each extra
checkpoint **once** on the gate vs the final. Skips cleanly if none were uploaded.

In [ ]:
import gc, numpy as np
globals().pop("ov",None); gc.collect()                     # free the final model (its metrics are in STAT)
if torch.cuda.is_available(): torch.cuda.empty_cache()

SELECT=[]
def eval_ckpt_once(local_dir,seed=4242):
    m=OmniVoice.from_pretrained(local_dir,device_map=("cuda:0" if DEVICE=="cuda" else "cpu"),dtype=torch.float16)
    torch.manual_seed(seed); np.random.seed(seed)
    rows=[]
    for p in probe_lines:
        a=m.generate(text=p["text"],language=LANG_CODE,ref_audio=REF_WAV_PATH,ref_text=REF_TEXT)
        rows.append(dict(**score_clip(np.asarray(a[0],dtype="float32"),OV_SR,p["text"])))
    allp=[pr for r in rows for pr in r["pairs"]]; v=lambda k:[r[k] for r in rows if r[k]==r[k]]
    del m; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return dict(tone_i2=_bal_tone_acc(allp),cer=float(np.mean(v("cer"))),
                ssim=float(np.mean(v("ssim"))),len_ratio=float(np.mean([r["len_ratio"] for r in rows])))

if not SUBCKPTS:
    print("no extra checkpoint-*/ subdirs on S3 — skipping (final ckpt-5000 variance is in §7).")
else:
    for sub in SUBCKPTS:
        ld=os.path.join(WORK,sub)
        pull_top_level(f"{CKPT_PREFIX}/{sub}",ld); materialize_ckpt(ld)
        try:
            r=eval_ckpt_once(ld); r["ckpt"]=sub; SELECT.append(r); print("  evaluated",sub,r)
        except Exception as e: print("  FAILED",sub,"->",type(e).__name__,e)
    SELECT.append(dict(ckpt="checkpoint-5000(final)",tone_i2=STAT["tone_i2"]["mean"],
                       cer=STAT["cer"]["mean"],ssim=STAT["ssim"]["mean"],len_ratio=STAT["len_ratio"]["mean"]))
    print(f"\n{'ckpt':24} {'tone_i2':>8} {'cer':>6} {'ssim':>6} {'lenR':>6}")
    for r in sorted(SELECT,key=lambda x:-x["tone_i2"]):
        print(f"{r['ckpt']:24} {r['tone_i2']:>8.3f} {r['cer']:>6.3f} {r['ssim']:>6.3f} {r['len_ratio']:>6.3f}")
    import json,datetime
    s3.put_object(Bucket=BUCKET,Key=f"{OUT_PREFIX}/checkpoint_selection.json",
                  Body=json.dumps(dict(date=datetime.date.today().isoformat(),rows=SELECT),ensure_ascii=False,indent=2).encode())
    print("-> s3://%s/%s/checkpoint_selection.json"%(BUCKET,OUT_PREFIX))

### Read-out
- **Variance (§7):** report `tone_i2 = mean ± std` (SEM = std/√N for the error bar on the mean). If the
  mean clears zero-shot 0.598 and chance 0.333 by more than the spread, the finetune's tone gain is real.
- **Cloning (§8):** `SSIM→held-out-voice` clearly **>** `SSIM→bible` with a per-voice majority = diversity
  intact. Converging / bible wins / <2 voices = collapse-or-inconclusive → mitigate with fewer epochs /
  multi-speaker data. SLR86 fallback is a *weaker* (in-training) test — re-run once NaijaVoices refs exist.
- **Selection (§9):** pick the best `tone_i2` checkpoint (+ ear), not necessarily the final.
- The **native ear** on `s3://…/omnivoice_ft_verify/full/` is still decisive.